# 🧩 Mini-Lab: HTTP Endpoint Basics

**Module: Delivery: CLI, API, and Debugging Lab 2** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** what an interface seam is and why it keeps your core logic independent of delivery mechanism
2. **Identify** boundary validation as the HTTP layer's responsibility — not the core's
3. **Recognize** how the same core function serves both a CLI and an HTTP endpoint unchanged

## Target Concepts

| Concept | Gloss |
|---------|-------|
| HTTP Endpoint Contract | A clear promise for an HTTP API endpoint: what request to send, what response to expect, and what errors and timing behavior are allowed. |
| Boundary Validation | Check user input and model output, the parts you do not control, so errors, abuse, and surprises are caught before they hit your core logic. |
| Interface Seam | A small typed contract between your app and an AI vendor. Your adapter returns your dataclass, never their SDK response object, so providers can be swapped without rewrites. |


## Why Interface Seams and Boundary Validation Matter

In `mini-build-cli` we built a CLI wrapper around a core function. The CLI parsed arguments, validated them (e.g., file not found → stderr + exit 1), and called the core.

An HTTP endpoint does **exactly the same job** — it's just a different **interface seam**:

```
┌──────────────┐        ┌────────────┐
│  CLI (argparse) │──────▶│            │
└──────────────┘        │  Core      │
┌──────────────┐        │  Logic     │
│  HTTP (FastAPI) │──────▶│ (pure fn)  │
└──────────────┘        └────────────┘
```

Each seam has one extra responsibility: **boundary validation** — checking that the incoming request is well-formed *before* the core ever sees it. The core trusts its inputs; the seam guarantees they're trustworthy.

| Responsibility | Where it lives |
|---------------|----------------|
| Parse & validate raw input | Interface seam (CLI or HTTP) |
| Business logic | Core function |
| Format output for the medium | Interface seam |

## Setup

In [1]:
import subprocess, sys, os, time, signal
from pathlib import Path

from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

We need **FastAPI** and **uvicorn** (the server that runs it), plus **httpx** to send test requests.

> If you already have these installed, the cell below is a no-op.

In [2]:
!pip install -q fastapi uvicorn httpx


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Step 1 — The Core Function (No HTTP, No CLI)

This is the **core logic** — a plain Python function. It knows nothing about argparse or FastAPI. Both interface seams will call it.

In [3]:
%%writefile core.py
"""Core logic — pure function, no I/O framework dependencies."""


def word_count(text: str) -> dict:
    """Count words in text. Returns a dict with the count.

    Precondition: `text` is a non-empty string (validated by the caller).
    """
    words = text.split()
    return {"word_count": len(words), "char_count": len(text)}

Writing core.py


Notice: no `import argparse`, no `from fastapi import ...`. The core is **interface-agnostic**. This is the interface seam principle in action.

## Step 2 — The HTTP Seam with Boundary Validation

> 🧩 **Http Endpoint Contract.** A clear promise for the request, the response, and the errors and timing that are allowed.
> Watch the next file: the endpoint is a thin contract over the core function.

The HTTP endpoint is a thin wrapper. Its only jobs are:

1. **Boundary validation** — reject bad requests *before* calling the core
2. **Call the core** — pass validated data
3. **Format the response** — return JSON with the right status code

FastAPI + Pydantic make boundary validation declarative: you define a model, and invalid requests get a `422` automatically.


In [4]:
%%writefile api.py
"""HTTP interface seam — thin wrapper around core logic."""
from fastapi import FastAPI
from pydantic import BaseModel, Field

from core import word_count  # import the core — not the other way around

app = FastAPI(title="Word Counter API")


# --- Boundary validation via Pydantic ---
class CountRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=10_000,
        description="The text to analyse (1–10 000 chars)",
    )


class CountResponse(BaseModel):
    word_count: int
    char_count: int


@app.post("/count", response_model=CountResponse)
def count_endpoint(req: CountRequest):
    # Core does the work; the seam just passes validated data through
    return word_count(req.text)

Writing api.py


### Where is the boundary validation?

It's in the `CountRequest` model:

| Rule | Pydantic Field | What happens on violation |
|------|---------------|---------------------------|
| `text` must be present | `...` (required) | 422 — missing field |
| `text` must not be empty | `min_length=1` | 422 — too short |
| `text` must not be huge | `max_length=10_000` | 422 — too long |
| `text` must be a string | type annotation `str` | 422 — wrong type |

The core function **never sees** invalid data — the seam blocks it at the boundary.

## Step 3 — Run the Server and Test

We'll start the uvicorn server as a background process, send requests with `httpx`, and then shut it down.

In [5]:
# Start the server in the background
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api:app", "--port", "8321"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Give the server a moment to start
time.sleep(3)
print(f"Server started (PID {server_proc.pid})")

Server started (PID 32232)


In [6]:
import httpx

BASE = "http://127.0.0.1:8321"

# --- Valid request ---
resp = httpx.post(f"{BASE}/count", json={"text": "The quick brown fox"})
print("Valid request:")
print(f"  Status: {resp.status_code}")
print(f"  Body:   {resp.json()}")

Valid request:
  Status: 200
  Body:   {'word_count': 4, 'char_count': 19}


### Boundary Validation in Action

Now let's send requests that **violate the boundary rules**. The seam rejects them with 422 before the core function is ever called.

In [7]:
# --- Missing field ---
resp = httpx.post(f"{BASE}/count", json={})
print(f"Missing 'text' → Status: {resp.status_code}")

# --- Empty string (violates min_length=1) ---
resp = httpx.post(f"{BASE}/count", json={"text": ""})
print(f"Empty text   → Status: {resp.status_code}")

# --- Too long (violates max_length=10_000) ---
resp = httpx.post(f"{BASE}/count", json={"text": "x" * 10_001})
print(f"Too long     → Status: {resp.status_code}")

Missing 'text' → Status: 422


Empty text   → Status: 422


Too long     → Status: 422


All three invalid requests received **422 Unprocessable Entity** — the HTTP-standard status for validation errors. The core's `word_count` function was never called for any of them.

## Step 4 — Same Core, Two Seams

To drive home the interface seam concept, recall the CLI from `mini-build-cli`. Here's how the same core serves both interfaces:

| Aspect | CLI seam (`argparse`) | HTTP seam (`FastAPI`) |
|--------|----------------------|-----------------------|
| Input parsing | `parser.parse_args()` | Pydantic `CountRequest` |
| Boundary validation | `if not path.exists(): sys.exit(1)` | `Field(min_length=1, max_length=...)` |
| Call to core | `word_count(text)` | `word_count(req.text)` |
| Output format | `print(count)` to stdout | JSON response `{"word_count": 4}` |
| Error signaling | Exit code 1 + stderr | HTTP 422 + JSON error body |

The core function is **identical** in both cases. The seam translates between the medium (terminal vs network) and the core.

## Cleanup

In [8]:
# Stop the server
server_proc.terminate()
server_proc.wait(timeout=5)
print("Server stopped.")

# Remove generated files
for f in ["core.py", "api.py"]:
    Path(f).unlink(missing_ok=True)
print("Cleaned up.")

Server stopped.
Cleaned up.


## 🎯 Summary

### Key Takeaways

1. **Interface seam** — a thin adapter layer (CLI or HTTP) that sits between the outside world and your core logic, keeping each side independent and swappable
2. **Boundary validation** — the seam's job is to reject bad input (missing fields, wrong types, out-of-range values) *before* the core function ever runs
3. **Pydantic models as boundaries** — in FastAPI, declaring a request model with `Field(min_length=..., max_length=...)` gives you automatic 422 responses for invalid requests
4. **Same core, different seams** — when your core is a plain function with no framework imports, wrapping it in CLI *or* HTTP is a small, reversible decision